# Space-Time Electric Potential Generator

Stage 02 maps the stage-01 current waveforms onto extracellular potentials along the MRG axon. This notebook shows the transformation for one repetition; the release data contain the full set of space-time arrays.


## Pipeline Map

For each CV/frequency bundle, stage 02:

1. Loads COMSOL +1 uA and -1 uA field profiles along the selected axon cutline.
2. Loads each stage-01 current waveform repetition.
3. Uses the sign of each current sample to select the matching COMSOL profile.
4. Writes `V(t, node)` arrays that become the NEURON extracellular drive in stage 03.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _bootstrap_src(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        src_dir = candidate / 'src'
        if src_dir.exists() and (candidate / 'data').exists():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = _bootstrap_src(Path.cwd())

from utils import require_data_files, data_subdir, ensure_src_on_path, find_repo_root
from figures import set_manuscript_style

REPO_ROOT = find_repo_root(REPO_ROOT)
SRC_DIR = ensure_src_on_path(REPO_ROOT)
require_data_files(
    REPO_ROOT,
    [
        'data/comsol/M101_1uA_bcs1_neg.txt',
        'data/comsol/M101_1uA_bcs1_pos.txt',
        'data/time-patterns/generated',
    ],
    context='02_SpaceTimeV_Generator',
)


from space_time import (
    DEFAULT_CV_VALUES,
    DEFAULT_FREQUENCIES_HZ,
    SpaceTimeConfig,
    load_comsol_profiles,
    load_time_pattern_repetition,
    map_current_to_spacetime,
    plot_comsol_profiles,
    plot_spacetime_construction,
    verify_spacetime_outputs,
)

set_manuscript_style(dpi=120)
pd.set_option('display.max_columns', None)

## Configure and Check Data

The data check below runs silently unless an expected stage-02 input or output is missing.


In [ ]:
comsol_dir = data_subdir(REPO_ROOT, 'comsol')
time_patterns_dir = data_subdir(REPO_ROOT, 'time_patterns')
space_time_dir = data_subdir(REPO_ROOT, 'space_time')
config = SpaceTimeConfig(
    cv_vals=DEFAULT_CV_VALUES,
    freq_vals=DEFAULT_FREQUENCIES_HZ,
    cutline=101,
    fs=20_000.0,
    comsol_dir=comsol_dir,
    time_pattern_dir=time_patterns_dir / 'generated',
    output_dir=space_time_dir / 'generated',
).resolved()

status = verify_spacetime_outputs(config)
if not status.complete:
    missing = list(status.missing_input_files) + list(status.missing_output_files)
    preview = chr(10).join(str(path) for path in missing[:8])
    raise FileNotFoundError('Stage-02 data are missing. First missing files:' + chr(10) + preview)

example_cv = 0.5
example_freq = 50

## Step 1: Load and Interpolate COMSOL Profiles

The COMSOL exports define extracellular voltage along the axon for unit-amplitude current sources. Stage 02 interpolates those fields onto the discrete MRG model nodes.


In [ ]:
profiles = load_comsol_profiles(config)

profile_summary = pd.DataFrame([
    {'phase': '-1 uA', 'min_mV': profiles.v_neg_mv.min(), 'max_mV': profiles.v_neg_mv.max()},
    {'phase': '+1 uA', 'min_mV': profiles.v_pos_mv.min(), 'max_mV': profiles.v_pos_mv.max()},
])

display(profile_summary.round(6))
fig, ax = plot_comsol_profiles(profiles)
display(fig)
plt.close(fig)

## Step 2: Load One Stage-01 Current Waveform

The walkthrough uses one generated repetition from the selected CV/frequency bundle. The full stage loops over every repetition in every bundle.


In [ ]:
time_s, rep_label, current = load_time_pattern_repetition(
    config=config,
    cv=example_cv,
    freq=example_freq,
    rep_label_substr='rep002',
)

pd.DataFrame([
    {
        'cv': example_cv,
        'freq_hz': example_freq,
        'rep_label': rep_label,
        'samples': current.size,
        'nonzero_samples': int(np.count_nonzero(current)),
    }
])

## Step 3: Map Current Samples to Space-Time Voltage

At each time sample, negative current is multiplied by the -1 uA COMSOL profile and positive current is multiplied by the +1 uA profile. Zero-current samples remain zero across nodes.


In [ ]:
v_example = map_current_to_spacetime(
    current=current,
    v_neg_mv=profiles.v_neg_mv,
    v_pos_mv=profiles.v_pos_mv,
)

pd.DataFrame([
    {
        'current_shape': current.shape,
        'v_ext_shape': v_example.shape,
        'nodes': profiles.node_numbers.size,
        'v_ext_min_mV': float(v_example.min()),
        'v_ext_max_mV': float(v_example.max()),
    }
])

## Step 4: Inspect the Space-Time Map

The top panel shows the current pulse train near one event. The bottom panel shows the extracellular potential across nodes over the same time window.


In [ ]:
fig, axes = plot_spacetime_construction(
    time_s=time_s,
    node_numbers=profiles.node_numbers,
    current=current,
    v_ext_mv=v_example,
    label=rep_label,
    window_ms=(0, 75),
    dpi=200,
)
display(fig)
plt.close(fig)

## Full Stage Output

The full stage applies this mapping to all 100 time pattern repetitions for every CV/frequency pair and saves one compressed `V(t, node)` bundle per pair. Stage 03 reads those bundles directly into the MRG/NEURON threshold search.
